In [4]:
"""
Stage 5 (PD vs HC, class-aware) - Final evaluation on held-out test set
========================================================================

For each panel size in {5, 7, 10, 12, 16}:
  1. Read consensus_hyperparams.csv from the matching Stage-4 output dir.
  2. Re-instantiate the same pipelines (RobustScaler -> classifier).
  3. Fit on the FULL train_postfeature.csv (binary PD vs HC).
     Sample weights = composite (sex x cell_type x class), applied to
     RBF, Linear, GaussianNB, ElasticNet. Tailored ENet / RFE gene
     subsets at >=80% fold-inclusion if those CSVs exist; otherwise
     full panel.
  4. Predict on the held-out test_postfeature.csv.
  5. Compute the FULL metric battery on BOTH train and test, plus the
     train - test gap (overfitting diagnostic). The battery includes
     OVERALL precision plus PER-CLASS precision / recall / F1 alongside
     the standard balanced-accuracy / AUROC / PR-AUC / MCC / Brier /
     Cohen's kappa / sensitivity / specificity / macro-F1.
  6. Q1-quality plots: ROC, PR, confusion matrices, metric heatmap,
     train-vs-test grouped bars + gap panel, per-class metrics bars.

Class encoding: NEG=HC=0 (minority), POS=PD=1 (majority).
"""

from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    cohen_kappa_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR    = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline")
PDHC_DIR        = PIPELINE_DIR / "PDHC new"
CLASS_AWARE_DIR = PDHC_DIR / "PDHC-class-aware"
SPLIT_DIR       = CLASS_AWARE_DIR / "Outputs" / "stage2_split_class_aware_PDHC"
STAGE4_BASE     = CLASS_AWARE_DIR / "Outputs"

META_PATH = PDHC_DIR / "Meta_data_PDHC.csv"

CONDITION_COL = "condition"
PANEL_SIZES   = [5, 10, 15, 25, 41]
INCLUSION_THR = 60.0
NEG_LABEL     = "HC"
POS_LABEL     = "PD"
SEED          = 42

INT_PARAMS = {"clf__n_neighbors", "rfe__n_features_to_select"}

sns.set_style("whitegrid", {"grid.linestyle": ":", "grid.alpha": 0.5})
plt.rcParams.update({
    "savefig.dpi":       300,
    "figure.dpi":        100,
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.titleweight":  "bold",
    "axes.labelsize":    10,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "legend.fontsize":   8.5,
    "legend.frameon":    False,
})

CLF_COLORS = {
    "mSVM-RBF":    "#0173B2",
    "mSVM-Linear": "#DE8F05",
    "mSVM-RFE":    "#029E73",
    "kNN":         "#CC78BC",
    "GaussianNB":  "#CA9161",
    "ElasticNet":  "#D55E00",
}
CLASS_NAMES = [NEG_LABEL, POS_LABEL]


# ============================ Data helpers ==================================
def load_meta(sample_ids):
    meta = pd.read_csv(META_PATH)
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    return meta.loc[sample_ids]


def to_binary_labels(y_str: np.ndarray) -> np.ndarray:
    return np.where(np.asarray(y_str) == NEG_LABEL, NEG_LABEL, POS_LABEL)


def encode_binary(y_str_binary: np.ndarray) -> np.ndarray:
    return np.where(y_str_binary == POS_LABEL, 1, 0).astype(int)


def load_pf(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0)
    looks_like_genes = df.index.astype(str).str.startswith("ENSG").any()
    if not looks_like_genes:
        df = df.T
    return df


def composite_sample_weights(meta: pd.DataFrame,
                             y_int: np.ndarray) -> np.ndarray:
    strata = meta["sex"].astype(str) + "_" + meta["cell_type"].astype(str)
    s_counts = strata.value_counts()
    n, k = len(strata), len(s_counts)
    w_conf = strata.map(lambda s: (n / k) / s_counts[s]).values.astype(float)

    classes, c_counts = np.unique(y_int, return_counts=True)
    n_cls = len(classes)
    class_w = {c: n / (n_cls * cnt) for c, cnt in zip(classes, c_counts)}
    w_class = np.array([class_w[y] for y in y_int], dtype=float)

    w = w_conf * w_class
    return w / w.mean()


# ============================ HP parsing ====================================
def _typed(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s in ("None", "nan", "NaN"):
        return None
    if s in ("True", "False"):
        return s == "True"
    if s.lower() in ("scale", "auto"):
        return s.lower()
    if s in ("uniform", "distance"):
        return s
    try:
        i = int(s)
        if str(i) == s:
            return i
    except ValueError:
        pass
    try:
        return float(s)
    except ValueError:
        return s


def consensus_hp_dict(consensus_df: pd.DataFrame) -> Dict[str, Dict]:
    out: Dict[str, Dict] = {}
    cdf = consensus_df.set_index("classifier")
    for clf in cdf.index:
        params = {}
        for col in cdf.columns:
            if col.endswith("__support_pct"):
                continue
            v = _typed(cdf.loc[clf, col])
            if v is None:
                continue
            params[col] = v
        out[clf] = params
    return out


def load_inclusion_genes(path: Path, threshold: float) -> List[str]:
    if not path.exists():
        return []
    df = pd.read_csv(path)
    if "fold_inclusion_pct" not in df.columns or "gene" not in df.columns:
        return []
    return df.loc[df["fold_inclusion_pct"] >= threshold, "gene"].tolist()


# ============================ Classifier factory ============================
def build_pipelines(seed: int = SEED) -> Dict[str, Dict]:
    return {
        "mSVM-RBF": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", SVC(kernel="rbf", probability=True,
                                          random_state=seed))]),
            "supports_sw": True,
        },
        "mSVM-Linear": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", SVC(kernel="linear", probability=True,
                                          random_state=seed))]),
            "supports_sw": True,
        },
        "mSVM-RFE": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("rfe", RFE(SVC(kernel="linear",
                                              probability=True,
                                              random_state=seed)))]),
            "supports_sw": False,
        },
        "kNN": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", KNeighborsClassifier())]),
            "supports_sw": False,
        },
        "GaussianNB": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", GaussianNB())]),
            "supports_sw": True,
        },
        "ElasticNet": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", LogisticRegression(
                                  penalty="elasticnet", solver="saga",
                                  max_iter=10000, tol=1e-4,
                                  random_state=seed))]),
            "supports_sw": True,
        },
    }


# ============================ Metrics =======================================
def evaluate_binary(y_true, y_pred, y_proba_pos) -> Dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan  # recall_PD
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan  # recall_HC

    prec_per_class = precision_score(y_true, y_pred, average=None,
                                     labels=[0, 1], zero_division=0)
    rec_per_class  = recall_score(y_true, y_pred, average=None,
                                  labels=[0, 1], zero_division=0)
    f1_per_class   = f1_score(y_true, y_pred, average=None,
                              labels=[0, 1], zero_division=0)

    return {
        "balanced_accuracy":  balanced_accuracy_score(y_true, y_pred),
        "macro_precision":    precision_score(y_true, y_pred,
                                              average="macro", zero_division=0),
        "macro_recall":       recall_score(y_true, y_pred,
                                           average="macro", zero_division=0),
        "macro_f1":           f1_score(y_true, y_pred, average="macro",
                                       zero_division=0),
        "auroc":              roc_auc_score(y_true, y_proba_pos),
        "pr_auc":             average_precision_score(y_true, y_proba_pos),
        "sensitivity":        sens,
        "specificity":        spec,
        "mcc":                matthews_corrcoef(y_true, y_pred),
        "brier":              brier_score_loss(y_true, y_proba_pos),
        "cohen_kappa":        cohen_kappa_score(y_true, y_pred),
        f"precision_{NEG_LABEL}": float(prec_per_class[0]),
        f"precision_{POS_LABEL}": float(prec_per_class[1]),
        f"recall_{NEG_LABEL}":    float(rec_per_class[0]),
        f"recall_{POS_LABEL}":    float(rec_per_class[1]),
        f"f1_{NEG_LABEL}":        float(f1_per_class[0]),
        f"f1_{POS_LABEL}":        float(f1_per_class[1]),
    }


# ============================ Plotting ======================================
def plot_roc(predictions: Dict, out_path: Path, title_suffix: str):
    fig, ax = plt.subplots(figsize=(8, 7))
    for clf_name, p in predictions.items():
        y_true = np.asarray(p["y_true"])
        y_pos  = np.asarray(p["y_proba_pos"])
        fpr, tpr, _ = roc_curve(y_true, y_pos)
        auc = roc_auc_score(y_true, y_pos)
        ax.plot(fpr, tpr, color=CLF_COLORS[clf_name], linewidth=1.8,
                label=f"{clf_name} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, linewidth=1)
    ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.01])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("1 - Specificity (FPR)")
    ax.set_ylabel("Sensitivity (TPR)")
    ax.set_title(f"ROC on held-out test - {title_suffix}", fontweight="bold")
    ax.legend(loc="lower right")
    fig.tight_layout(); fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_pr(predictions: Dict, out_path: Path, title_suffix: str):
    fig, ax = plt.subplots(figsize=(8, 7))
    for clf_name, p in predictions.items():
        y_true = np.asarray(p["y_true"])
        y_pos  = np.asarray(p["y_proba_pos"])
        prec, rec, _ = precision_recall_curve(y_true, y_pos)
        ap = average_precision_score(y_true, y_pos)
        ax.plot(rec, prec, color=CLF_COLORS[clf_name], linewidth=1.8,
                label=f"{clf_name} (AP={ap:.3f})")
    prevalence = np.mean([np.mean(np.asarray(p["y_true"]))
                          for p in predictions.values()])
    ax.axhline(prevalence, color="gray", linestyle="--", linewidth=1, alpha=0.7,
               label=f"prevalence = {prevalence:.2f}")
    ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.01])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"Precision-Recall on held-out test - {title_suffix}",
                 fontweight="bold")
    ax.legend(loc="lower left")
    fig.tight_layout(); fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_confusion_grid(predictions: Dict, out_path: Path, title_suffix: str):
    n = len(predictions)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.2 * rows),
                             constrained_layout=True)
    axes = np.array(axes).ravel()

    for ax, (clf_name, p) in zip(axes, predictions.items()):
        y_true = np.asarray(p["y_true"])
        y_pred = np.asarray(p["y_pred"])
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

        sns.heatmap(cm_norm, annot=cm.astype(int), fmt="d",
                    cmap="Blues", vmin=0, vmax=1, cbar=False, square=True,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    annot_kws={"fontsize": 11, "fontweight": "bold"},
                    ax=ax)
        for i in range(2):
            for j in range(2):
                ax.text(j + 0.5, i + 0.78, f"{cm_norm[i, j]*100:.0f}%",
                        ha="center", va="center", fontsize=8.5,
                        color="white" if cm_norm[i, j] > 0.5 else "#444")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(clf_name)

    for ax in axes[len(predictions):]:
        ax.axis("off")
    fig.suptitle(f"Confusion matrices on held-out test - {title_suffix}\n"
                 "(counts; row-percentage below)", fontweight="bold")
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_metric_heatmap(metrics_df: pd.DataFrame, out_path: Path,
                        title_suffix: str):
    cols = ["balanced_accuracy", "macro_precision", "macro_recall", "macro_f1",
            "auroc", "pr_auc", "sensitivity", "specificity", "mcc",
            "cohen_kappa", "brier",
            f"precision_{NEG_LABEL}", f"precision_{POS_LABEL}",
            f"recall_{NEG_LABEL}",    f"recall_{POS_LABEL}",
            f"f1_{NEG_LABEL}",        f"f1_{POS_LABEL}"]
    pretty = ["Bal. acc.", "macro-Prec.", "macro-Rec.", "macro-F1",
              "AUROC", "PR-AUC", "Sens. (PD)", "Spec. (HC)", "MCC",
              "Cohen's κ", "Brier (↓)",
              f"Prec. {NEG_LABEL}", f"Prec. {POS_LABEL}",
              f"Rec. {NEG_LABEL}",  f"Rec. {POS_LABEL}",
              f"F1 {NEG_LABEL}",    f"F1 {POS_LABEL}"]
    M = metrics_df.set_index("classifier")[cols].copy()
    M.columns = pretty

    norm_M = M.copy()
    norm_M["Brier (↓)"] = 1 - norm_M["Brier (↓)"].clip(0, 1)

    fig, ax = plt.subplots(figsize=(15, 4.8))
    sns.heatmap(norm_M, annot=M.round(3), fmt="", cmap="RdYlGn",
                vmin=0, vmax=1, linewidths=0.5, linecolor="white",
                cbar_kws={"shrink": 0.7}, annot_kws={"fontsize": 8.5}, ax=ax)
    ax.set_title(f"Full metric battery - {title_suffix} "
                 "(green = better; Brier inverted for colouring only)",
                 fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    fig.tight_layout(); fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_per_class_metrics(metrics_df: pd.DataFrame, out_path: Path,
                           title_suffix: str):
    rows = []
    for _, r in metrics_df.iterrows():
        for cls in (NEG_LABEL, POS_LABEL):
            rows.append({"classifier": r["classifier"], "class": cls,
                         "metric": "Precision",
                         "score": float(r[f"precision_{cls}"])})
            rows.append({"classifier": r["classifier"], "class": cls,
                         "metric": "Recall",
                         "score": float(r[f"recall_{cls}"])})
            rows.append({"classifier": r["classifier"], "class": cls,
                         "metric": "F1",
                         "score": float(r[f"f1_{cls}"])})
    long_df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5),
                             constrained_layout=True, sharey=True)
    for ax, m in zip(axes, ["Precision", "Recall", "F1"]):
        sub = long_df[long_df["metric"] == m]
        sns.barplot(data=sub, x="classifier", y="score", hue="class",
                    palette={NEG_LABEL: "#d62728", POS_LABEL: "#1f77b4"},
                    ax=ax, edgecolor="white")
        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", fontsize=7.5, padding=1)
        ax.set_ylim([0, 1.05]); ax.set_ylabel("Score")
        ax.set_xlabel(""); ax.set_title(m, fontweight="bold")
        plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
        if ax.get_legend(): ax.get_legend().set_title("Class")
    fig.suptitle(f"Per-class metrics on held-out test - {title_suffix}",
                 fontweight="bold")
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_train_vs_test(train_df: pd.DataFrame, test_df: pd.DataFrame,
                       out_path: Path, title_suffix: str):
    headline = ["balanced_accuracy", "macro_precision", "macro_recall",
                "macro_f1", "auroc", "pr_auc", "mcc", "cohen_kappa"]
    pretty = {"balanced_accuracy": "Bal. acc.",
              "macro_precision":   "macro-Prec.",
              "macro_recall":      "macro-Rec.",
              "macro_f1":          "macro-F1",
              "auroc":             "AUROC",
              "pr_auc":            "PR-AUC",
              "mcc":               "MCC",
              "cohen_kappa":       "Cohen's κ"}

    tr = train_df.set_index("classifier")[headline].copy()
    te =  test_df.set_index("classifier")[headline].copy()
    gap = tr - te

    metric_order = [pretty[h] for h in headline]
    clf_order = list(CLF_COLORS)

    fig = plt.figure(figsize=(16, 9))
    gs = fig.add_gridspec(2, 1, height_ratios=[2.0, 1.0], hspace=0.35)
    ax_top = fig.add_subplot(gs[0])
    ax_bot = fig.add_subplot(gs[1])

    width = 0.06
    x = np.arange(len(metric_order))
    for ci, clf in enumerate(clf_order):
        offset = (ci - len(clf_order) / 2 + 0.5) * width * 2
        tr_vals = [float(tr.loc[clf, h]) for h in headline]
        te_vals = [float(te.loc[clf, h]) for h in headline]
        ax_top.bar(x + offset - width / 2, tr_vals, width=width,
                   color=CLF_COLORS[clf], edgecolor="white", linewidth=0.5,
                   alpha=0.95)
        ax_top.bar(x + offset + width / 2, te_vals, width=width,
                   color=CLF_COLORS[clf], edgecolor="white", linewidth=0.5,
                   hatch="///", alpha=0.95)

    ax_top.set_xticks(x); ax_top.set_xticklabels(metric_order)
    ax_top.set_ylim([min(0, min(tr.values.min(), te.values.min()) - 0.1), 1.05])
    ax_top.axhline(0, color="black", linewidth=0.6)
    ax_top.set_ylabel("Score"); ax_top.set_xlabel("")
    ax_top.set_title(f"Train vs Test - {title_suffix} "
                     "(solid = Train, hatched = Test)", fontweight="bold")

    handles = [Patch(facecolor=CLF_COLORS[c], edgecolor="white", label=c)
               for c in clf_order]
    handles += [Patch(facecolor="white", edgecolor="black", label="Train"),
                Patch(facecolor="white", edgecolor="black",
                      hatch="///", label="Test")]
    ax_top.legend(handles=handles, loc="upper left",
                  bbox_to_anchor=(1.01, 1.0))

    gap_long = []
    for clf in tr.index:
        for metric in headline:
            gap_long.append({"classifier": clf,
                             "metric": pretty[metric],
                             "gap": float(gap.loc[clf, metric])})
    gap_long_df = pd.DataFrame(gap_long)
    sns.barplot(data=gap_long_df, x="metric", y="gap",
                hue="classifier", hue_order=clf_order,
                order=metric_order, palette=CLF_COLORS,
                edgecolor="white", linewidth=0.6, ax=ax_bot)
    for container in ax_bot.containers:
        ax_bot.bar_label(container, fmt="%+.2f", padding=2,
                         fontsize=7, label_type="edge")
    ax_bot.axhline(0, color="black", linewidth=0.8)
    ax_bot.set_ylabel("Train - Test gap"); ax_bot.set_xlabel("")
    ax_bot.set_title("Generalisation gap (positive = train > test = overfit)",
                     fontweight="bold")
    if ax_bot.get_legend():
        ax_bot.get_legend().remove()

    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


# ============================ Per-panel run =================================
def run_for_panel(k_top: int):
    s4_dir = STAGE4_BASE / f"stage4_classifiers_PDHC_top{k_top}"
    out_dir = STAGE5_BASE / f"stage5_final_evaluation_PDHC_top{k_top}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 72)
    print(f"PANEL SIZE: top {k_top} - output: {out_dir.name}")
    print("=" * 72)

    train_df = load_pf(s4_dir / "train_postfeature.csv")
    test_df  = load_pf(s4_dir / "test_postfeature.csv")
    print(f"Train: {train_df.shape[1]} samples x {train_df.shape[0]} genes")
    print(f"Test : {test_df.shape[1]} samples x {test_df.shape[0]} genes")

    full_genes = train_df.index.tolist()

    meta_train = load_meta(train_df.columns)
    meta_test  = load_meta(test_df.columns)

    y_train_str = meta_train[CONDITION_COL].values
    y_test_str  = meta_test[CONDITION_COL].values
    y_train_bin = to_binary_labels(y_train_str)
    y_test_bin  = to_binary_labels(y_test_str)
    y_train     = encode_binary(y_train_bin)
    y_test      = encode_binary(y_test_bin)

    print(f"Train class balance: HC={int((y_train==0).sum())}, "
          f"PD={int((y_train==1).sum())}")
    print(f"Test  class balance: HC={int((y_test==0).sum())}, "
          f"PD={int((y_test==1).sum())}")

    X_train_full = train_df.T.values.astype(np.float64)
    X_test_full  = test_df.T.values.astype(np.float64)
    sw_train = composite_sample_weights(meta_train, y_train)

    consensus = pd.read_csv(s4_dir / "consensus_hyperparams.csv")
    hp_dict   = consensus_hp_dict(consensus)

    enet_inc = s4_dir / "enet_fold_inclusion.csv"
    rfe_inc  = s4_dir / "rfe_fold_inclusion.csv"
    enet_genes = [g for g in load_inclusion_genes(enet_inc, INCLUSION_THR)
                  if g in full_genes]
    rfe_genes  = [g for g in load_inclusion_genes(rfe_inc,  INCLUSION_THR)
                  if g in full_genes]
    tailored = {"ElasticNet": enet_genes, "mSVM-RFE": rfe_genes}
    print(f"\nTailored gene subsets at fold_inclusion_pct >= {INCLUSION_THR}%:")
    for k, gs in tailored.items():
        print(f"  {k}: {len(gs)} genes")

    pipelines = build_pipelines(SEED)
    predictions: Dict[str, Dict] = {}
    metrics_rows, train_metrics_rows, gap_rows = [], [], []
    hp_rows, gene_subset_rows = [], []

    for clf_name, cfg in pipelines.items():
        if clf_name in tailored and len(tailored[clf_name]) >= 2:
            genes_used = tailored[clf_name]
            subset_label = f"tailored ({len(genes_used)} >= {INCLUSION_THR}%)"
        else:
            if clf_name in tailored:
                print(f"  [{clf_name}] WARN: tailored set empty/<2; "
                      f"using full panel")
            genes_used = full_genes
            subset_label = f"full panel ({len(genes_used)} genes)"

        gene_idx = [full_genes.index(g) for g in genes_used]
        X_tr = X_train_full[:, gene_idx]
        X_te = X_test_full[:,  gene_idx]
        gene_subset_rows.append({
            "classifier":  clf_name,
            "subset_kind": subset_label,
            "n_genes":     len(genes_used),
            "genes":       ";".join(genes_used),
        })

        params = hp_dict.get(clf_name, {})
        valid_keys = set(cfg["pipe"].get_params().keys())
        applied = {k: v for k, v in params.items() if k in valid_keys}

        for ip in INT_PARAMS & set(applied.keys()):
            if applied[ip] is not None:
                applied[ip] = int(float(applied[ip]))

        if clf_name == "mSVM-RFE" and "rfe__n_features_to_select" in applied:
            requested = int(applied["rfe__n_features_to_select"])
            applied["rfe__n_features_to_select"] = max(1,
                min(requested, len(genes_used)))
            if applied["rfe__n_features_to_select"] != requested:
                print(f"  [{clf_name}] clamped n_features_to_select "
                      f"{requested} -> {applied['rfe__n_features_to_select']}")

        cfg["pipe"].set_params(**applied)
        hp_rows.append({"classifier": clf_name,
                        "applied_params": str(applied),
                        "subset_kind": subset_label,
                        "n_genes": len(genes_used)})

        fit_kwargs = {}
        if cfg["supports_sw"]:
            fit_kwargs["clf__sample_weight"] = sw_train

        cfg["pipe"].fit(X_tr, y_train, **fit_kwargs)

        y_train_pred  = cfg["pipe"].predict(X_tr)
        y_train_proba = cfg["pipe"].predict_proba(X_tr)[:, 1]
        m_train = evaluate_binary(y_train, y_train_pred, y_train_proba)
        m_train["classifier"] = clf_name
        train_metrics_rows.append(m_train)

        y_pred  = cfg["pipe"].predict(X_te)
        y_proba_pos = cfg["pipe"].predict_proba(X_te)[:, 1]
        m = evaluate_binary(y_test, y_pred, y_proba_pos)
        m["classifier"] = clf_name
        metrics_rows.append(m)

        gap_row = {"classifier": clf_name}
        for k, v_train in m_train.items():
            if k == "classifier":
                continue
            v_test = m.get(k)
            if isinstance(v_train, (int, float)) and isinstance(v_test, (int, float)):
                gap_row[f"{k}_train"] = v_train
                gap_row[f"{k}_test"]  = v_test
                gap_row[f"{k}_gap"]   = v_train - v_test
        gap_rows.append(gap_row)

        predictions[clf_name] = {
            "y_true":      y_test.tolist(),
            "y_pred":      y_pred.tolist(),
            "y_proba_pos": y_proba_pos.tolist(),
        }

        print(f"\n[{clf_name}] {subset_label} | applied: {applied}")
        print(f"  TRAIN BA={m_train['balanced_accuracy']:.3f} "
              f"AUROC={m_train['auroc']:.3f} "
              f"macro-Prec={m_train['macro_precision']:.3f} "
              f"F1={m_train['macro_f1']:.3f}")
        print(f"  TEST  BA={m['balanced_accuracy']:.3f} "
              f"AUROC={m['auroc']:.3f} "
              f"macro-Prec={m['macro_precision']:.3f} "
              f"F1={m['macro_f1']:.3f} "
              f"MCC={m['mcc']:.3f}")
        print(f"  TEST per-class - "
              f"{NEG_LABEL}: P={m[f'precision_{NEG_LABEL}']:.3f} "
              f"R={m[f'recall_{NEG_LABEL}']:.3f} "
              f"F1={m[f'f1_{NEG_LABEL}']:.3f} | "
              f"{POS_LABEL}: P={m[f'precision_{POS_LABEL}']:.3f} "
              f"R={m[f'recall_{POS_LABEL}']:.3f} "
              f"F1={m[f'f1_{POS_LABEL}']:.3f}")

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df = metrics_df[["classifier"] + [c for c in metrics_df.columns
                                              if c != "classifier"]]
    metrics_df.to_csv(out_dir / "final_test_metrics.csv", index=False)

    train_metrics_df = pd.DataFrame(train_metrics_rows)
    train_metrics_df = train_metrics_df[["classifier"]
        + [c for c in train_metrics_df.columns if c != "classifier"]]
    train_metrics_df.to_csv(out_dir / "final_train_metrics.csv", index=False)

    pd.DataFrame(gap_rows).to_csv(out_dir / "train_test_gap.csv", index=False)
    pd.DataFrame(hp_rows).to_csv(out_dir / "consensus_hp_used.csv",
                                 index=False)
    pd.DataFrame(gene_subset_rows).to_csv(
        out_dir / "gene_subsets_per_classifier.csv", index=False)

    per_class_rows = []
    for _, r in metrics_df.iterrows():
        for cls in (NEG_LABEL, POS_LABEL):
            per_class_rows.append({
                "classifier": r["classifier"],
                "class":      cls,
                "precision":  float(r[f"precision_{cls}"]),
                "recall":     float(r[f"recall_{cls}"]),
                "f1":         float(r[f"f1_{cls}"]),
            })
    pd.DataFrame(per_class_rows).to_csv(
        out_dir / "per_class_metrics_test.csv", index=False)

    pred_rows = []
    for clf_name, p in predictions.items():
        for sid, yt, yp, yprob in zip(test_df.columns, p["y_true"],
                                       p["y_pred"], p["y_proba_pos"]):
            pred_rows.append({"classifier": clf_name, "sample_id": sid,
                              "y_true":  CLASS_NAMES[yt],
                              "y_pred":  CLASS_NAMES[yp],
                              f"proba_{POS_LABEL}": yprob,
                              f"proba_{NEG_LABEL}": 1.0 - yprob})
    pd.DataFrame(pred_rows).to_csv(
        out_dir / "final_test_predictions.csv", index=False)

    title_suffix = f"PD vs HC (class-aware), top {k_top}"
    plot_roc(predictions, out_dir / "roc_curves_test.png", title_suffix)
    plot_pr(predictions,  out_dir / "pr_curves_test.png",  title_suffix)
    plot_confusion_grid(predictions, out_dir / "confusion_matrices_test.png",
                        title_suffix)
    plot_metric_heatmap(metrics_df, out_dir / "metric_heatmap_test.png",
                        title_suffix)
    plot_per_class_metrics(metrics_df, out_dir / "per_class_metrics_test.png",
                           title_suffix)
    plot_train_vs_test(train_metrics_df, metrics_df,
                       out_dir / "train_vs_test_metrics.png", title_suffix)

    print(f"\nOutputs in: {out_dir}")


def main():
    print("=" * 72)
    print("STAGE 5 (PD vs HC, class-aware) - Final evaluation on held-out test")
    print(f"Panel sizes: {PANEL_SIZES}")
    print("=" * 72)
    for k in PANEL_SIZES:
        run_for_panel(k)


if __name__ == "__main__":
    main()


STAGE 5 (PD vs HC, class-aware) - Final evaluation on held-out test
Panel sizes: [5, 10, 15, 25, 41]

PANEL SIZE: top 5 - output: stage5_final_evaluation_PDHC_top5
Train: 108 samples x 5 genes
Test : 47 samples x 5 genes
Train class balance: HC=36, PD=72
Test  class balance: HC=17, PD=30

Tailored gene subsets at fold_inclusion_pct >= 60.0%:
  ElasticNet: 5 genes
  mSVM-RFE: 2 genes

[mSVM-RBF] full panel (5 genes) | applied: {'clf__C': 10.0, 'clf__gamma': 'scale'}
  TRAIN BA=0.868 AUROC=0.084 macro-Prec=0.851 F1=0.858
  TEST  BA=0.677 AUROC=0.318 macro-Prec=0.677 F1=0.677 MCC=0.355
  TEST per-class - HC: P=0.588 R=0.588 F1=0.588 | PD: P=0.767 R=0.767 F1=0.767

[mSVM-Linear] full panel (5 genes) | applied: {'clf__C': 1.0}
  TRAIN BA=0.590 AUROC=0.656 macro-Prec=0.581 F1=0.558
  TEST  BA=0.507 AUROC=0.575 macro-Prec=0.507 F1=0.468 MCC=0.014
  TEST per-class - HC: P=0.367 R=0.647 F1=0.468 | PD: P=0.647 R=0.367 F1=0.468

[mSVM-RFE] tailored (2 >= 60.0%) | applied: {'rfe__estimator__C': 10